Embedding

How many document objects will be created from x no. of pages in a pdf for example is loader-dependent. A page can be a Document, depending on the loader/configuration. A Document can then be split into multiple chunks. Another loader/configuration may produce a single Document for the entire file or another structure.

An embedding model converts text into a vector of numbers that represents its semantic meaning.
That vector is then stored in a vector database and used for similarity search

Open AI embedding model:

Common OpenAI embedding models

Model	                 Dimensions	            General idea

text-embedding-3-small	  1536	    Cheaper, good general-purpose choice

text-embedding-3-large	   3072 by default	   Higher-quality embeddings,  useful when retrieval accuracy matters

text-embedding-ada-002	   1536	           Older/legacy model

In [ ]:

from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_community.embeddings import HuggingFaceEmbeddings
from dotenv import load_dotenv

load_dotenv()

embeddings= OpenAIEmbeddings(model="text-embedding-3-small")

# single text
text = "This is a sample text to be embedded."
embedding = embeddings.embed_query(text)
print(f"Embedding for single text: {embedding}")


In [ ]:

from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_ollama import OllamaEmbeddings
from dotenv import load_dotenv

load_dotenv()

#HuggingFaceEmbeddings and OllamaEmbeddings are two different LangChain integration classes, and each class defines its own constructor parameter names.

#the class expects a parameter called model_name
embeddings1= HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

embeddings2=OllamaEmbeddings(model="llama2-7b-embedding-q4_0")

In the context of embeddings and RAG, normalizing a vector means converting it so that its length (magnitude) becomes 1, while keeping its direction the same.

NumPy (Numerical Python) is a Python library mainly used for working with numbers, arrays, vectors, matrices, and mathematical operations efficiently.

In [ ]:

from langchain_openai.embeddings import OpenAIEmbeddings
from dotenv import load_dotenv
import numpy as np
load_dotenv()

embeddings= OpenAIEmbeddings(model="text-embedding-3-small")

def basic_embeddings():
    #single text
    text = "What is Machine Learning?"
    single_embedding=embeddings.embed_query(text)
    print(f"Vector dimensions: {len(single_embedding)}")
    print(f"First 5 values: {single_embedding [:5]}")
    print(f"Vector norm: {np.linalg.norm(single_embedding):.4f}")

    #np.linalg is a NumPy module for linear algebra.
    #norm() calculates the magnitude/length of a vector.
    #print it with 4 digits after the decimal

if __name__=="__main__":
    basic_embeddings()


Normalizing a vector prevents longer documents from having larger vectors just because they have more content.

Short text → [1536 numbers]
Long text  → [1536 numbers]

The number of dimensions doesn't become larger.

What can differ is the magnitude (norm) of the vector, depending on the embedding model and how it represents the input.

If your similarity calculation is affected by vector magnitude, the longer document could appear more important just because its vector is larger.These two vectors point in the same direction.But the second vector has a much larger magnitude simply because it contains more content.

Before normalization:

Short → smaller magnitude
Long  → larger magnitude

After normalization:

Short → magnitude = 1
Long  → magnitude = 1

Their magnitudes are removed as a factor, while their direction is preserved.
Now the comparison focuses much more on direction, which is what cosine similarity measures.
Normalization doesn't make a long document shorter. It makes the vector's length equal to 1, so similarity isn't unfairly influenced by vector magnitude.

In [ ]:
from langchain_openai.embeddings import OpenAIEmbeddings
from dotenv import load_dotenv
import numpy as np

def batch_embeddings():
    text = [
    "What is Machine Learning?",
    "Explain the concept of overfitting in ML.",
    "How does a neural network work?",
    ]

    batch_embeddings=OpenAIEmbeddings.embed_documents(text)
    for i,emb in enumerate(batch_embeddings):
         print(f"Vector dimensions: {len(emb)}")
         print(f"First 5 values: {emb[:5]}")
         print(f"Vector norm: {np.linalg.norm(emb):.4f}")
            

In [ ]:
from langchain_openai.embeddings import OpenAIEmbeddings
from dotenv import load_dotenv
import numpy as np


embeddings=OpenAIEmbeddings(model="text-embedding-3-small")

texts = [
    "What is Machine Learning?",
    "Explain the concept of overfitting in ML.",
    "How does a neural network work?",
    ]
#if text is a list of multiple texts,produces multiple embedding vectors — one vector for each text.
doc_vectors=embeddings.embed_documents(texts)

query="This is a single line text."

query_vector=embeddings.embed_query(query)


# Compute cosine similarity
def cosine_similarity(vec1, vec2):
    return np.dot(vec1, vec2) / (
        np.linalg.norm(vec1) * np.linalg.norm(vec2)
    )

similarities = [
    cosine_similarity(query_vector, doc_vec) for doc_vec in doc_vectors
    ]
# Rank documents by similarity
ranked_docs = sorted(
    zip(texts, similarities),
    key=lambda x: x[1],
    reverse=True
)

for doc, score in ranked_docs:
    print(f"Document: {doc}")
    print(f"Similarity score: {score:.4f}")

Embedding Caching:
To avoid redundant API Calls
Calling an embedding model and inferring it will incur costs.

In [ ]:
#This code is demonstrating embedding caching: the first time you create an embedding, it calls OpenAI; the next time you ask for the same text, it retrieves the already-created embedding from local storage instead of calling the API again.

from langchain_classic.embeddings.cache import CacheBackedEmbeddings # an embedding wrapper that can check a cache before generating a new embedding.
from langchain_classic.storage import LocalFileStore #gives you a simple storage mechanism using files on your computer.
import tempfile
import numpy as np

#Your computer's filesystem stores cached embedding data
with tempfile.TemporaryDirectory() as tempdir: #creates a temporary folder and Store the path of that temporary directory in the variable tempdir
    store=LocalFileStore(root_path=tempdir)
    #root_path= is a keyword argument.
    #Telling LocalFileStore to Use this temporary directory(tempdir) as your storage location.

    #from_bytes_store This is a class method. It's a way of constructing a CacheBackedEmbeddings object using a storage system that stores the cached embedding data as bytes.
    cached_embeddings=CacheBackedEmbeddings.from_bytes_store(
        underlying_embeddings=OpenAIEmbeddings(model="text-embedding-3-small"), #The actual embedding model that will be used if the embedding isn't already cached.
        document_embedding_cache=store, #Where should I look for/store cached document embeddings?
        namespace="excercise" #This provides a namespace/prefix to separate these cached embeddings from other cached data.
    )
    #The key point is: you don't explicitly call a store() or save() method in your code. CacheBackedEmbeddings does it internally. triggers the cache lookup + possible storage both.


    text="what is reinforcement learning" #That's simply the input whose embedding you want.

    # First call -hits API
    print("First call (API):") 
    vectors1 = cached_embeddings.embed_documents ([text]) #embed_documents() expects a list of texts. ["what is reinforcement learning"] that's a list containing one document. embed.documents() returns a list of embedding vectors. [[0.123, -0.456, 0.789, ...]] The outer list represents the documents.The inner list is the embedding vector
    print(f" Embedded {len (vectors1)} documents")

    # Second call from cache -
    print("\nSecond call (Cache):")
    vectors2 = cached_embeddings.embed_documents ([text])
    print(f" Embedded {len (vectors2)} documents")
    

    # Verify same results
    print(f"\nSame vectors: {np.allclose (vectors1[0], vectors2[0])}")
    #to check whether two vectors are approximately equal.Because we're working with floating-point numbers.

CacheBackedEmbeddings
                         │
                  Check local cache
                    ↙          ↘
                FOUND          NOT FOUND
                  ↓                ↓
             Return vector    OpenAIEmbeddings
                                  ↓
                            OpenAI API